In [20]:
import pandas as pd

In [21]:
df = pd.read_csv('data/cropdata.csv')
df.head()

,crop ID,soil_type,Seedling Stage,MOI,temp,humidity,result
0,Wheat,Black Soil,Germination,1,25,80.0,1
1,Wheat,Black Soil,Germination,2,26,77.0,1
2,Wheat,Black Soil,Germination,3,27,74.0,1
3,Wheat,Black Soil,Germination,4,28,71.0,1
4,Wheat,Black Soil,Germination,5,29,68.0,1


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16411 entries, 0 to 16410
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   crop ID         16411 non-null  object 
 1   soil_type       16411 non-null  object 
 2   Seedling Stage  16411 non-null  object 
 3   MOI             16411 non-null  int64  
 4   temp            16411 non-null  int64  
 5   humidity        16411 non-null  float64
 6   result          16411 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 897.6+ KB


In [23]:
# soil types
df['soil_type'].unique()

array(['Black Soil', 'Alluvial Soil', 'Sandy Soil', 'Red Soil',
       'Clay Soil', 'Loam Soil', 'Chalky Soil'], dtype=object)

In [24]:
# drop rows with 'Black Soil', 'Chalky Soil', 'Red Soil' and missing values
df_filtered = df[~df['soil_type'].isin(['Black Soil', 'Chalky Soil', 'Red Soil'])].drop(columns=['crop ID', 'Seedling Stage']).reset_index(drop=True).copy()
print(df_filtered['soil_type'].unique())
df_filtered.head()

['Alluvial Soil' 'Sandy Soil' 'Clay Soil' 'Loam Soil']


,soil_type,MOI,temp,humidity,result
0,Alluvial Soil,1,33,56.0,1
1,Alluvial Soil,2,34,53.0,1
2,Alluvial Soil,3,35,50.0,1
3,Alluvial Soil,4,36,47.0,1
4,Alluvial Soil,5,37,44.0,1


In [25]:
# drop rows with 'result' values of 2
df_filtered = df_filtered[df_filtered['result'] != 2].reset_index(drop=True).copy()
print(df_filtered['result'].unique())

[1 0]


In [26]:
label_maping = {
    'Alluvial Soil': 0,
    'Sandy Soil': 1,
    'Clay Soil': 2,
    'Loam Soil': 3
}
df_filtered['soil_type'] = df_filtered['soil_type'].map(label_maping)
df_filtered.head()


,soil_type,MOI,temp,humidity,result
0,0,1,33,56.0,1
1,0,2,34,53.0,1
2,0,3,35,50.0,1
3,0,4,36,47.0,1
4,0,5,37,44.0,1


In [27]:
X = df_filtered.drop(columns=['result'])
y = df_filtered['result']

In [28]:
y.unique()

array([1, 0])

## Modelling

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

In [30]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
X_test.head()

,soil_type,MOI,temp,humidity
9309,2,31,24,80.0
7277,3,71,31,58.0
2908,1,13,18,86.0
8491,1,69,16,88.0
3815,1,94,44,23.0


In [32]:
y_test.head()

9309    0
7277    0
2908    0
8491    0
3815    0
Name: result, dtype: int64

### SVM

In [33]:
from sklearn.svm import SVC

# Scale the features (SVM is sensitive to feature scales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the SVM model
svm_model = SVC(kernel='rbf', random_state=42, gamma='scale')
svm_model.fit(X_train_scaled, y_train)

# Make predictions using the test set
y_pred = svm_model.predict(X_test_scaled)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.952288218111003

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.97      0.96      1246
           1       0.95      0.93      0.94       808

    accuracy                           0.95      2054
   macro avg       0.95      0.95      0.95      2054
weighted avg       0.95      0.95      0.95      2054



### Decision Tree

In [34]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train the Decision Tree model
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)

# Make predictions using the test set
y_pred_dt = dt_model.predict(X_test)

# Evaluate the model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
print("Accuracy:", accuracy_dt)
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))

Accuracy: 0.9664070107108081

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.96      0.97      1246
           1       0.94      0.97      0.96       808

    accuracy                           0.97      2054
   macro avg       0.96      0.97      0.97      2054
weighted avg       0.97      0.97      0.97      2054



## Conversion

In [35]:
import pickle

In [36]:
with open("saved/scaler.pkl", 'wb') as f:
  pickle.dump(scaler, f)

In [37]:
with open("saved/svm_model.pkl", 'wb') as f:
  pickle.dump(svm_model, f)

In [38]:
with open("saved/dt.pkl", 'wb') as f:
  pickle.dump(dt_model, f)

# Conclusion